Uniform Cost Search: Nairobi to Kisumu
This notebook continues the highway-network exercise by applying Uniform Cost Search (UCS) to the same towns and path costs used in Part 1.

Goal: find the minimum-cost route for the trailer from Nairobi to Kisumu.

UCS always expands the frontier node with the lowest cumulative path cost g(n). Because all road costs are positive, the first time Kisumu is removed from the priority queue, the path is optimal.

In [ ]:
# Weighted highway graph from Part 1.
# Each road is represented in both directions because the Part 1 description
# states that the road links can be travelled both ways at the same cost.

kenya_highway_network = {
    "Nairobi": [("Thika", 42), ("Naivasha", 92), ("Narok", 140)],
    "Thika": [("Nairobi", 42), ("Nyeri", 100)],
    "Nyeri": [("Thika", 100), ("Nyahururu", 70)],
    "Nyahururu": [("Nyeri", 70), ("Nakuru", 65)],
    "Naivasha": [("Nairobi", 92), ("Nakuru", 63)],
    "Narok": [("Nairobi", 140), ("Kisii", 190)],
    "Nakuru": [
        ("Naivasha", 63),
        ("Nyahururu", 65),
        ("Kericho", 115),
        ("Eldoret", 155),
    ],
    "Kericho": [("Nakuru", 115), ("Kisumu", 80)],
    "Eldoret": [("Nakuru", 155), ("Kisumu", 130)],
    "Kisii": [("Narok", 190), ("Kisumu", 115)],
    "Kisumu": [("Kericho", 80), ("Eldoret", 130), ("Kisii", 115)],
}

kenya_highway_network

{'Nairobi': [('Thika', 42), ('Naivasha', 92), ('Narok', 140)],
 'Thika': [('Nairobi', 42), ('Nyeri', 100)],
 'Nyeri': [('Thika', 100), ('Nyahururu', 70)],
 'Nyahururu': [('Nyeri', 70), ('Nakuru', 65)],
 'Naivasha': [('Nairobi', 92), ('Nakuru', 63)],
 'Narok': [('Nairobi', 140), ('Kisii', 190)],
 'Nakuru': [('Naivasha', 63),
  ('Nyahururu', 65),
  ('Kericho', 115),
  ('Eldoret', 155)],
 'Kericho': [('Nakuru', 115), ('Kisumu', 80)],
 'Eldoret': [('Nakuru', 155), ('Kisumu', 130)],
 'Kisii': [('Narok', 190), ('Kisumu', 115)],
 'Kisumu': [('Kericho', 80), ('Eldoret', 130), ('Kisii', 115)]}

In [ ]:
import heapq
import itertools

def uniform_cost_search(graph, start, goal):
    """Uniform Cost Search with a step-by-step traversal trace."""

    counter = itertools.count()

    # Entries: (cumulative_cost, tie_breaker, state, path, leg_costs)
    frontier = [(0, next(counter), start, [start], [])]

    # Lowest known cost to each state.
    best_cost = {start: 0}

    # States whose cheapest path has already been expanded.
    expanded = set()

    selection_order = []
    step = 0

    print("=" * 78)
    print(f"UNIFORM COST SEARCH: {start} -> {goal}")
    print("=" * 78)

    while frontier:
        cost, _, state, path, leg_costs = heapq.heappop(frontier)

        # Ignore an outdated queue entry if a cheaper route was found later.
        if cost != best_cost.get(state):
            continue

        if state in expanded:
            continue

        step += 1
        selection_order.append((state, cost))

        print(f"\nStep {step}: select {state} with g(n) = {cost} km")

        # In UCS, the goal is accepted when it is selected as the cheapest
        # node on the frontier, not when it is first generated.
        if state == goal:
            print(f"Goal reached: {goal}")
            return {
                "path": path,
                "leg_costs": leg_costs,
                "total_cost": cost,
                "selection_order": selection_order,
                "expanded_before_goal": len(expanded),
            }

        expanded.add(state)

        for neighbor, edge_cost in graph.get(state, []):
            if neighbor in expanded:
                continue

            new_cost = cost + edge_cost

            if new_cost < best_cost.get(neighbor, float("inf")):
                best_cost[neighbor] = new_cost

                heapq.heappush(
                    frontier,
                    (
                        new_cost,
                        next(counter),
                        neighbor,
                        path + [neighbor],
                        leg_costs + [edge_cost],
                    ),
                )

                print(
                    f"  add/update {neighbor}: "
                    f"{cost} + {edge_cost} = {new_cost} km"
                )

        # Show only current best frontier entries.
        current_frontier = sorted(
            (c, s)
            for c, _, s, _, _ in frontier
            if s not in expanded and c == best_cost.get(s)
        )

        if current_frontier:
            print(
                "  frontier:",
                ", ".join(f"{state}({cost})" for cost, state in current_frontier),
            )

    return None

In [ ]:
result = uniform_cost_search(
    kenya_highway_network,
    start="Nairobi",
    goal="Kisumu"
)

print("\n" + "=" * 78)
print("FINAL RESULT")
print("=" * 78)

if result is None:
    print("No route found.")
else:
    path = result["path"]
    leg_costs = result["leg_costs"]

    print("Optimal path:")
    print(" -> ".join(path))

    print("\nPath-cost calculation:")
    print(" + ".join(map(str, leg_costs)), "=", result["total_cost"], "km")

    print("\nUCS selection order:")
    print(
        " -> ".join(
            f"{state}({cost})"
            for state, cost in result["selection_order"]
        )
    )

    print(
        f"\nNodes expanded before the goal: "
        f"{result['expanded_before_goal']}"
    )

    assert path == [
        "Nairobi", "Naivasha", "Nakuru", "Kericho", "Kisumu"
    ]
    assert result["total_cost"] == 350

UNIFORM COST SEARCH: Nairobi -> Kisumu

Step 1: select Nairobi with g(n) = 0 km
  add/update Thika: 0 + 42 = 42 km
  add/update Naivasha: 0 + 92 = 92 km
  add/update Narok: 0 + 140 = 140 km
  frontier: Thika(42), Naivasha(92), Narok(140)

Step 2: select Thika with g(n) = 42 km
  add/update Nyeri: 42 + 100 = 142 km
  frontier: Naivasha(92), Narok(140), Nyeri(142)

Step 3: select Naivasha with g(n) = 92 km
  add/update Nakuru: 92 + 63 = 155 km
  frontier: Narok(140), Nyeri(142), Nakuru(155)

Step 4: select Narok with g(n) = 140 km
  add/update Kisii: 140 + 190 = 330 km
  frontier: Nyeri(142), Nakuru(155), Kisii(330)

Step 5: select Nyeri with g(n) = 142 km
  add/update Nyahururu: 142 + 70 = 212 km
  frontier: Nakuru(155), Nyahururu(212), Kisii(330)

Step 6: select Nakuru with g(n) = 155 km
  add/update Kericho: 155 + 115 = 270 km
  add/update Eldoret: 155 + 155 = 310 km
  frontier: Nyahururu(212), Kericho(270), Eldoret(310), Kisii(330)

Step 7: select Nyahururu with g(n) = 212 km
  front